Yargi

In [ ]:
!pip install yargy pymorphy2

PHONE

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df


def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])


def extend_phone_span(text: str, start: int, end: int):

    tail = text[end:end + 50]

    paren_ext = re.match(
        r"\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\)",
        tail,
        flags=re.IGNORECASE
    )

    ext = re.match(
        r"\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}",
        tail,
        flags=re.IGNORECASE
    )

    if paren_ext:
        end += paren_ext.end()
    elif ext:
        end += ext.end()

    tail = text[end:end + 40]

    list_tail = re.match(
        r"(?:\s*[,/]\s*\d{2}){1,4}",
        tail
    )

    if list_tail:
        end += list_tail.end()

    return start, end


def is_bad_phone_candidate(text: str, start: int, end: int, source: str = "") -> bool:
    value = text[start:end]
    digits = re.sub(r"\D", "", value)
    stripped = value.strip()

    left_context_15 = text[max(0, start - 15):start].lower()
    left_context_40 = text[max(0, start - 40):start].lower()

    phone_context = re.search(
        r"(тел\.?|телефон|моб\.?|номер телефона|контактный телефон|контактный номер|для связи)\s*[:\-]?\s*$",
        left_context_40,
        flags=re.IGNORECASE
    )

    bad_context = re.search(
        r"(номер документа|регистрационный номер|номер обращения|id клиента|user_id|id|инн|р/с|счет|номер счета|идентификатор)\s*[:=]?\s*$",
        left_context_40,
        flags=re.IGNORECASE
    )

    if "в/ч" in left_context_15:
        return True

    if bad_context and not phone_context:
        return True

    if stripped.isdigit() and len(digits) < 10:
        return True

    if stripped.isdigit() and len(digits) >= 11:
        if not re.match(r"^(?:\+?7|8)\d{10}$", stripped) and not phone_context:
            return True

    if re.fullmatch(r"\d{1,3}-\d{2}-\d{2}(?:\s*(?:[,/]\s*\d{2}){1,4})?", stripped):
        if not phone_context:
            return True

    if re.search(
        r"(код|артикул|номер договора|номер сч[её]та|номер заявки|код заказа|код операции)",
        left_context_40,
        flags=re.IGNORECASE
    ):
        if not re.search(
            r"(тел\.?|телефон|моб\.?|для связи)",
            left_context_40,
            flags=re.IGNORECASE
        ):
            return True

    return False



class YargyPhoneDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def digits_pred(pattern):
            return custom(lambda t: bool(re.fullmatch(pattern, token_value(t))))

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(
                        pattern,
                        token_value(t),
                        flags=re.IGNORECASE | re.VERBOSE
                    )
                )
            )

        DIG1 = digits_pred(r"\d{1}")
        DIG2 = digits_pred(r"\d{2}")
        DIG3 = digits_pred(r"\d{3}")
        DIG4 = digits_pred(r"\d{4}")
        DIG7 = digits_pred(r"\d{7}")
        DIG10 = digits_pred(r"\d{10}")
        DIG11 = digits_pred(r"\d{11}")
        DIG1_2 = digits_pred(r"\d{1,2}")
        DIG2_3 = digits_pred(r"\d{2,3}")
        DIG2_4 = digits_pred(r"\d{2,4}")
        DIG1_6 = digits_pred(r"\d{1,6}")
        DIG6 = digits_pred(r"\d{6}")

        PHONE_COMPACT_TOKEN = token_pred(
            r"""
            (?:
                \+?7[\.\-\s]?\(?\d{3}\)?[\.\-\s]?\d{3}[\.\-\s]?\d{2}[\.\-\s]?\d{2}
                |
                8[\.\-\s]?\(?\d{3}\)?[\.\-\s]?\d{3}[\.\-\s]?\d{2}[\.\-\s]?\d{2}
                |
                8\(\d{3}\)\d{7}
                |
                \+7\(\d{3}\)\d{7}
                |
                \d{3}-\d{3}-\d{3}\s?\d{2}
                |
                \d{4}-\d{7}
            )
            """
        )

        CODE_PAREN_TOKEN = token_pred(r"\(\d{3}\)")

        SEP = or_(rule(eq("-")), rule(eq(".")))

        PLUS7 = rule(eq("+"), eq("7"))

        PREFIX = or_(
            PLUS7,
            rule(eq("7")),
            rule(eq("8")),
        )

        CODE_PAREN = or_(
            rule(eq("("), DIG3, eq(")")),
            rule(CODE_PAREN_TOKEN),
        )

        CODE = or_(
            rule(DIG3),
            CODE_PAREN,
        )

        PHONE_SINGLE_TOKEN = rule(PHONE_COMPACT_TOKEN)

        PHONE_PLUS_SOLID = rule(eq("+"), DIG11)

        PHONE_BROKEN_PAREN = rule(
            PLUS7,
            eq("("),
            DIG10,
        )

        PHONE_DOT_FULL = rule(
            PLUS7,
            eq("."),
            DIG3,
            eq("."),
            DIG3,
            eq("."),
            DIG2,
            eq("."),
            DIG2,
        )

        PHONE_SOLID_11 = rule(DIG11)

        PHONE_8_3_4_3 = rule(
            eq("8"),
            DIG3,
            DIG4,
            DIG3,
        )

        PHONE_3_3_3_2 = rule(
            DIG3,
            eq("-"),
            DIG3,
            eq("-"),
            DIG3,
            DIG2,
        )

        PHONE_WITH_PREFIX = rule(
            PREFIX,
            SEP.optional(),
            CODE,
            SEP.optional(),
            DIG2_4,
            SEP.optional(),
            DIG1_2,
            SEP.optional(),
            DIG2_3,
        )

        PHONE_PREFIX_CODE_7DIG = rule(
            PREFIX,
            CODE,
            DIG7,
        )

        PHONE_PAREN_NO_PREFIX = rule(
            CODE_PAREN,
            DIG3,
            eq("-"),
            DIG2,
            eq("-"),
            DIG2,
        )

        PHONE_NO_PREFIX_GROUPED = rule(
            DIG3,
            DIG3,
            DIG2,
            DIG2,
        )

        PHONE_WITHOUT_PREFIX = or_(
            rule(DIG10),
            rule(
                CODE,
                SEP.optional(),
                DIG3,
                SEP.optional(),
                DIG2,
                SEP.optional(),
                DIG2,
            ),
        )

        LOCAL_PHONE = or_(
            rule(DIG3, eq("-"), DIG2, eq("-"), DIG2),
            rule(DIG2, eq("-"), DIG2, eq("-"), DIG2),
            rule(DIG1, eq("-"), DIG2, eq("-"), DIG2),
        )

        PHONE_WEIRD = or_(
            rule(DIG4, eq("-"), DIG7),
            rule(eq("8"), DIG3, SEP, DIG7),
            rule(eq("8"), DIG6, DIG2, DIG2),
            rule(PLUS7, DIG10),
        )

        LIST_TAIL = or_(
            rule(eq("/"), DIG2),
            rule(eq(","), DIG2),
            rule(eq(","), DIG2, eq(","), DIG2),
        )

        EXT_MARKER = or_(
            rule(caseless("доб")),
            rule(caseless("вн")),
            rule(caseless("ext")),
            rule(eq("#")),
        )

        EXTENSION = or_(
            rule(EXT_MARKER, eq(".").optional(), DIG1_6),
            rule(eq("("), caseless("вн"), eq(".").optional(), DIG1_6, eq(")")),
        )

        BASE_PHONE = or_(
            PHONE_SINGLE_TOKEN,
            PHONE_PLUS_SOLID,
            PHONE_BROKEN_PAREN,
            PHONE_DOT_FULL,
            PHONE_PREFIX_CODE_7DIG,
            PHONE_8_3_4_3,
            PHONE_3_3_3_2,
            PHONE_PAREN_NO_PREFIX,
            PHONE_NO_PREFIX_GROUPED,
            PHONE_WITH_PREFIX,
            PHONE_WEIRD,
            PHONE_WITHOUT_PREFIX,
            PHONE_SOLID_11,
            LOCAL_PHONE,
        )

        PHONE_WORDS = rule(
            caseless("восемь"),
            caseless("девятьсот"),
            caseless("двадцать"),
            caseless("шесть"),
            caseless("сто"),
            caseless("двадцать"),
            caseless("три"),
            caseless("сорок"),
            caseless("пять"),
            caseless("шестьдесят"),
            caseless("семь"),
        )

        PHONE = or_(
            rule(
                BASE_PHONE,
                LIST_TAIL.optional(),
                EXTENSION.optional(),
            ),
            PHONE_WORDS,
        )

        self.parser = Parser(PHONE)

        self.phone_regexes = [
            r"(?<!\d)(?:\+7|8)\.\d{3}\.\d{3}\.\d{2}\.\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\(\d{3}\)\d{7}(?!\d)",

            r"(?<!\d)(?:\+?7|8)\d{10}(?!\d)",

            r"(?<!\d)(?:\+7|8)[-\s]\d{3}[-\s]\d{3}[-\s]\d{2}[-\s]\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\s*\(\d{3}\)\s*\d{3}[\s-]\d{2}[\s-]\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\s*\(\d{3}\)\s*\d{7}(?!\d)",

            r"(?<!\d)(?:\+7|8|7)\s+\d{3}\s+\d{3}\s+\d{2}\s+\d{2}(?!\d)",

            r"(?<!\d)\d{3}\s+\d{3}\s+\d{2}\s+\d{2}(?!\d)",

            r"(?<!\d)\(\d{3}\)\s*\d{3}-\d{2}-\d{2}(?!\d)",

            r"(?<!\d)\+?7\d{10}(?:\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}|\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\))?(?!\d)",

            r"(?<!\d)8\s*\(\d{3}\)\s*\d{7}(?:\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}|\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\))?(?!\d)",

            r"(?<![\d-])\d{3}-\d{2}-\d{2}(?:\s*(?:[,/]\s*\d{2}){1,4})?(?![\d-])",

            r"(?<!\d)\d{1,2}-\d{2}-\d{2}(?!\d)",
        ]

    def _add_entity(self, entities, text, start, end, source):
        if is_bad_phone_candidate(text, start, end, source=source):
            return

        start, end = extend_phone_span(text, start, end)

        entities.append({
            "start": start,
            "end": end,
            "label": "PHONE",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities=entities,
                text=text,
                start=match.span.start,
                end=match.span.stop,
                source="yargy",
            )

        for pattern in self.phone_regexes:
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                self._add_entity(
                    entities=entities,
                    text=text,
                    start=m.start(),
                    end=m.end(),
                    source="regex_phone",
                )

        return resolve_overlaps(entities)



def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def run_yargy_phone_experiment(path: str):
    df = load_dataset(path)
    detector = YargyPhoneDetector()

    texts = df["text"].tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-PHONE-plus-regex",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []
    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df



def print_phone_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "PHONE"]
        pred = [e for e in row["predictions"] if e["label"] == "PHONE"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → PHONE')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → PHONE')

    return errors_df


path = "/content/test_v1 - test_final_300_v3.csv (20).csv"

summary, class_table, predictions_df = run_yargy_phone_experiment(path)

predictions_df.to_csv("yargy_phone_predictions.csv", index=False)
class_table.to_csv("yargy_phone_metrics.csv", index=False)

errors_df = print_phone_errors(predictions_df)
errors_df.to_csv("yargy_phone_errors.csv", index=False)

In [ ]:
def evaluate_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)

    detector = YargyPhoneDetector()

    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        phone_preds = [p for p in preds if p["label"] == "PHONE"]

        if phone_preds:
            texts_with_fp += 1
            total_fp += len(phone_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(phone_preds),
                "false_positives": phone_preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(f'FP: "{e["text"]}" [{e["start"]}:{e["end"]}] → {e["label"]}')

    return summary, errors_df

In [ ]:
tricky_summary, tricky_errors = evaluate_negative_or_tricky(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv",
    name="tricky"
)

tricky_errors.to_csv("yargy_phone_tricky_errors.csv", index=False)

In [ ]:
nopd_summary, nopd_errors = evaluate_negative_or_tricky(
    "/content/no_pll_labeled - no_pll_labeled.csv (7).csv",
    name="no_pd"
)

nopd_errors.to_csv("yargy_phone_no_pd_errors.csv", index=False)

ID

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]



def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df



def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])



class YargyIdDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def digits_pred(pattern):
            return custom(lambda t: bool(re.fullmatch(pattern, token_value(t))))

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(pattern, token_value(t), flags=re.IGNORECASE)
                )
            )

        DIG1_2 = digits_pred(r"\d{1,2}")
        DIG2 = digits_pred(r"\d{2}")
        DIG3 = digits_pred(r"\d{3}")
        DIG4 = digits_pred(r"\d{4}")
        DIG1_20 = digits_pred(r"\d{1,20}")
        DIG11 = digits_pred(r"\d{11}")
        DIG12 = digits_pred(r"\d{12}")

        RU_LETTERS = token_pred(r"[А-ЯЁA-Z]{1,6}")
        ALNUM = token_pred(r"[A-ZА-ЯЁ0-9]{2,20}")

        NUM_SIGN = eq("№")

        ID_MARKER = or_(
            rule(caseless("ID")),
            rule(caseless("id")),
            rule(caseless("user_id")),
        )

        ID_CONTEXT_MARKER = or_(
            rule(caseless("ID"), caseless("клиента")),
            rule(caseless("id"), caseless("клиента")),
            rule(caseless("клиента")),
            rule(caseless("регистрационный"), caseless("ID")),
            rule(caseless("регистрационный"), caseless("id")),
            rule(caseless("идентификатор")),
        )

        DOC_MARKER = or_(
            rule(caseless("договор")),
            rule(caseless("договором")),
            rule(caseless("контракт")),
            rule(caseless("контрактом")),
            rule(caseless("регистрационный"), caseless("контракт")),
            rule(caseless("заявка")),
            rule(caseless("заявки")),
            rule(caseless("заявок")),
            rule(caseless("номер"), caseless("заявки")),
            rule(caseless("номер"), caseless("заявок")),
            rule(caseless("номера"), caseless("заявок")),
            rule(caseless("регистрационные"), caseless("заявки")),
            rule(caseless("идентификаторы"), caseless("заявок")),
            rule(caseless("идентификатор"), caseless("заявки")),
            rule(caseless("документ")),
            rule(caseless("идентификатор")),
            rule(caseless("номер"), caseless("документа")),
            rule(caseless("номер"), caseless("договора")),
            rule(caseless("номер"), caseless("контракта")),
            rule(caseless("номер"), caseless("обращения")),
            rule(caseless("регистрационный"), caseless("номер")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("заявки")),
        )

        INN_MARKER = rule(caseless("ИНН"))
        SNILS_MARKER = rule(caseless("СНИЛС"))

        ACCOUNT_MARKER = or_(
            rule(caseless("счет")),
            rule(caseless("счёт")),
            rule(caseless("номер"), caseless("счета")),
            rule(caseless("номер"), caseless("счёта")),
            rule(caseless("номером"), caseless("счета")),
            rule(caseless("номером"), caseless("счёта")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("счета")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("счёта")),
            rule(caseless("р"), eq("/"), caseless("с")),
        )

        SIMPLE_ID = or_(
            rule(ID_MARKER, eq(":").optional(), eq("=").optional(), DIG1_20),
            rule(ID_CONTEXT_MARKER, eq(":").optional(), NUM_SIGN.optional(), DIG1_20),
            rule(NUM_SIGN, DIG1_20),
        )

        DOC_NUMBER_ALNUM = or_(
            rule(RU_LETTERS, eq("-"), DIG4, eq("/"), DIG1_2, eq("-"), DIG1_2),
            rule(DIG4, eq("-"), DIG1_20),
            rule(DIG3, eq("-"), DIG3, eq("-"), ALNUM),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, DIG2),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, eq("/"), DIG1_2),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, eq("-"), DIG2),
            rule(DIG1_2, eq("-"), DIG3, eq("-"), RU_LETTERS),
            rule(DIG1_20, eq("-"), DIG1_20),
            rule(DIG1_20, eq("-"), RU_LETTERS),
            rule(DIG1_20),
        )

        DOC_ID = or_(
            rule(DOC_MARKER, eq(":").optional(), NUM_SIGN.optional(), DOC_NUMBER_ALNUM),
            rule(NUM_SIGN, DOC_NUMBER_ALNUM),
        )

        INN_ID = rule(INN_MARKER, eq(":").optional(), DIG12)

        SNILS_FORMATTED = rule(
            SNILS_MARKER,
            eq(":").optional(),
            DIG3,
            eq("-"),
            DIG3,
            eq("-"),
            DIG3,
            DIG2,
        )

        SNILS_SOLID_WITH_MARKER = rule(
            SNILS_MARKER,
            eq(":").optional(),
            DIG11,
        )

        GOVERNMENT_ID = or_(
            INN_ID,
            SNILS_FORMATTED,
            SNILS_SOLID_WITH_MARKER,
        )

        BANK_ACCOUNT = rule(
            ACCOUNT_MARKER,
            eq(":").optional(),
            DIG1_20,
        )

        ENUM_WITH_MARKER = or_(
            rule(caseless("заявки"), eq(":").optional(), DIG1_20),
            rule(caseless("заявка"), eq(":").optional(), DIG1_20),
            rule(caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("номера"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("номер"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("идентификаторы"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("регистрационные"), caseless("заявки"), eq(":").optional(), DIG1_20),
        )

        ID = or_(
            GOVERNMENT_ID,
            BANK_ACCOUNT,
            DOC_ID,
            SIMPLE_ID,
            ENUM_WITH_MARKER,
        )

        self.parser = Parser(ID)

    def trim_id_span(self, text: str, start: int, end: int):
        value = text[start:end]

        patterns = [
            r"^(?:ID|id|user_id)\s*[:=]?\s*",
            r"^(?:клиента)\s*:?\s*№?\s*",
            r"^(?:ИНН|СНИЛС)\s*:?\s*",
            r"^(?:№)\s*",

            r"^(?:ом)\s*:?\s*№?\s*",
            r"^(?:а)\s+(?=\d)",

            r"^(?:документ)\s*:?\s*№?\s*",
            r"^(?:договор|договором|договора|контракт|контрактом|контракта|регистрационный\s+контракт)\s*:?\s*№?\s*",
            r"^(?:заявка|заявки|заявок|номер\s+заявки|номер\s+заявок|номера\s+заявок|регистрационные\s+заявки|идентификаторы\s+заявок|идентификатор\s+заявки|регистрационный\s+номер\s+заявки)\s*:?\s*№?\s*",
            r"^(?:номер\s+документа|номер\s+договора|номер\s+контракта|номер\s+обращения|регистрационный\s+номер)\s*:?\s*№?\s*",
            r"^(?:ID\s+клиента|id\s+клиента|регистрационный\s+ID|регистрационный\s+id|идентификатор)\s*:?\s*№?\s*",
            r"^(?:счет|счёт|счета|счёта|номером\s+счета|номером\s+счёта|номер\s+счета|номер\s+счёта|регистрационный\s+номер\s+счета|регистрационный\s+номер\s+счёта|р/с)\s*:?\s*",
        ]

        changed = True
        while changed:
            changed = False
            value = text[start:end]

            for pattern in patterns:
                m = re.match(pattern, value, flags=re.IGNORECASE)
                if m:
                    start += m.end()
                    changed = True
                    break

        return start, end

    def extend_id_span(self, text: str, start: int, end: int):
        value = text[start:end]

        if re.fullmatch(r"\d{1,4}", value):
            tail = text[end:end + 40]

            m = re.match(
                r"(?:-\d{1,6})?(?:-[A-ZА-ЯЁ0-9]{1,20})?(?:/\d{1,4})?",
                tail,
                flags=re.IGNORECASE
            )

            if m:
                end += m.end()

        return start, end

    def split_enumerated_ids(self, text: str, start: int, end: int):
        value = text[start:end]

        if re.fullmatch(r"\d{1,20}(?:\s*,\s*\d{1,20})+", value):
            return [
                (start + m.start(), start + m.end())
                for m in re.finditer(r"\d{1,20}", value)
            ]

        return [(start, end)]

    def _is_bad_id_candidate(self, text: str, start: int, end: int) -> bool:
        value = text[start:end]
        value_lower = value.lower()
        digits = re.sub(r"\D", "", value)

        left_context_80 = text[max(0, start - 80):start].lower()
        right_context_30 = text[end:end + 30]

        phone_context = re.search(
            r"(тел\.?|телефон|моб\.?|контактный номер|номер телефона|для связи)\s*[:\-]?\s*$",
            left_context_80,
            flags=re.IGNORECASE
        )

        if phone_context:
            return True

        bad_left_context = re.search(
            r"(школ[аеуыи]?|поликлиник[аеуыи]?|изолятор[аеуыи]?|общежити[ея]|"
            r"дом|кв\.?|квартира|офис|оф\.?|стр\.?|корп\.?|"
            r"рейс|шаг|этап)\s*$",
            left_context_80,
            flags=re.IGNORECASE
        )

        if bad_left_context:
            return True

        if re.fullmatch(r"\d{1,3}-\d{3}", value) and re.match(
            r"-\d{2,3}-\d{2}-\d{2}",
            right_context_30
        ):
            return True

        has_marker = re.search(
            r"(id|user_id|инн|снилс|№|договор|контракт|заявк|сч[её]т|р/с|номер|регистрационный|идентификатор|клиента)",
            value_lower + " " + left_context_80,
            flags=re.IGNORECASE
        )

        if len(digits) <= 3 and not has_marker:
            return True

        return False

    def _add_entity(self, entities, text, start, end, source):
        if self._is_bad_id_candidate(text, start, end):
            return

        start, end = self.trim_id_span(text, start, end)
        start, end = self.trim_id_span(text, start, end)
        start, end = self.extend_id_span(text, start, end)

        if start >= end:
            return

        for sub_start, sub_end in self.split_enumerated_ids(text, start, end):
            if self._is_bad_id_candidate(text, sub_start, sub_end):
                continue

            entities.append({
                "start": sub_start,
                "end": sub_end,
                "label": "ID",
                "text": text[sub_start:sub_end],
                "source": source,
            })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities,
                text,
                match.span.start,
                match.span.stop,
                "yargy_id",
            )

        for m in re.finditer(
            r"(?:идентификаторы|номера|номер|регистрационные)?\s*заяв(?:ок|ки)\s*:\s*((?:\d{1,20}\s*,\s*)+\d{1,20})",
            text,
            flags=re.IGNORECASE
        ):
            nums_start = m.start(1)
            nums_part = m.group(1)

            for num in re.finditer(r"\d{1,20}", nums_part):
                sub_start = nums_start + num.start()
                sub_end = nums_start + num.end()

                entities.append({
                    "start": sub_start,
                    "end": sub_end,
                    "label": "ID",
                    "text": text[sub_start:sub_end],
                    "source": "yargy_id_enum",
                })

        for m in re.finditer(
            r"(?:идентификаторы|идентификатор)\s*:\s*((?:\d{1,20}\s*,\s*)+\d{1,20})",
            text,
            flags=re.IGNORECASE
        ):
            nums_start = m.start(1)
            nums_part = m.group(1)

            for num in re.finditer(r"\d{1,20}", nums_part):
                sub_start = nums_start + num.start()
                sub_end = nums_start + num.end()

                entities.append({
                    "start": sub_start,
                    "end": sub_end,
                    "label": "ID",
                    "text": text[sub_start:sub_end],
                    "source": "yargy_id_enum",
                })

        return resolve_overlaps(entities)


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def run_yargy_id_experiment(path: str):
    df = load_dataset(path)
    detector = YargyIdDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-ID-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []
    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df



def print_id_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "ID"]
        pred = [e for e in row["predictions"] if e["label"] == "ID"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → ID')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → ID')

    return errors_df



path = "/content/test_v1 - test_final_300_v3.csv (20).csv"

summary, class_table, predictions_df = run_yargy_id_experiment(path)

predictions_df.to_csv("yargy_id_predictions.csv", index=False)
class_table.to_csv("yargy_id_metrics.csv", index=False)

errors_df = print_id_errors(predictions_df)
errors_df.to_csv("yargy_id_errors.csv", index=False)

Другие выборки:

In [ ]:
def evaluate_id_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)


    detector = YargyIdDetector()

    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        id_preds = [p for p in preds if p["label"] == "ID"]

        if id_preds:
            texts_with_fp += 1
            total_fp += len(id_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(id_preds),
                "false_positives": id_preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(f'FP: "{e["text"]}" [{e["start"]}:{e["end"]}] → {e["label"]}')

    return summary, errors_df

In [ ]:
tricky_id_summary, tricky_id_errors = evaluate_id_negative_or_tricky(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv",
    name="tricky"
)

tricky_id_errors.to_csv("yargy_id_tricky_errors.csv", index=False)

In [ ]:
nopd_id_summary, nopd_id_errors = evaluate_id_negative_or_tricky(
    "/content/no_pll_labeled - no_pll_labeled.csv (7).csv",
    name="no_pd"
)

nopd_id_errors.to_csv("yargy_id_no_pd_errors.csv", index=False)

EMAIL

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]



def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df



def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])



class YargyEmailDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(pattern, token_value(t), flags=re.IGNORECASE)
                )
            )

        LOCAL_PART = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
        )

        DOMAIN_PART = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
        )

        TLD = token_pred(
            r"[A-ZА-ЯЁ0-9]{2,20}"
        )

        EMAIL_FULL_TOKEN = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
            r"@"
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
            r"(?:\.[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*|,[A-ZА-ЯЁ]{2,15})+"
        )

        DOT_OR_COMMA = or_(rule(eq(".")), rule(eq(",")))

        EMAIL_BASIC = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL_MULTI_DOMAIN = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL_THREE_DOMAIN = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL = or_(
            rule(EMAIL_FULL_TOKEN),
            EMAIL_THREE_DOMAIN,
            EMAIL_MULTI_DOMAIN,
            EMAIL_BASIC,
        )

        self.parser = Parser(EMAIL)

        self.email_regex = re.compile(
            r"[A-ZА-ЯЁ0-9]+(?:\s*[._+\-]\s*[A-ZА-ЯЁ0-9]+)*"
            r"\s*@\s*"
            r"[A-ZА-ЯЁ0-9]+(?:\s*-\s*[A-ZА-ЯЁ0-9]+)*"
            r"(?:"
            r"\s*\.\s*[A-ZА-ЯЁ0-9]+(?:\s*-\s*[A-ZА-ЯЁ0-9]+)*"
            r"|,[A-ZА-ЯЁ]{2,15}"
            r")+",
            flags=re.IGNORECASE
        )

    def normalize_email_candidate(self, value: str) -> str:
        compact = re.sub(r"\s+", "", value)
        compact = compact.replace(",", ".")
        return compact

    def trim_email_span(self, text: str, start: int, end: int):
        value = text[start:end]

        m = re.search(r",\s+", value)
        if m:
            end = start + m.start()

        while end > start and text[end - 1] in ".,;:!?)]}":
            end -= 1

        return start, end

    def is_bad_email_candidate(self, text: str, start: int, end: int) -> bool:
        start, end = self.trim_email_span(text, start, end)
        value = text[start:end]
        compact = self.normalize_email_candidate(value)

        if start > 0:
            prev_char = text[start - 1]
            if re.match(r"[A-Za-zА-Яа-яЁё0-9._+\-]", prev_char):
                return True

        if end < len(text):
            next_char = text[end]
            if re.match(r"[A-Za-zА-Яа-яЁё0-9_\-]", next_char):
                return True

        if not re.fullmatch(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
            r"@"
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
            r"(?:\.[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*)+",
            compact,
            flags=re.IGNORECASE
        ):
            return True

        return False

    def extend_email_span(self, text: str, start: int, end: int):
        window_start = max(0, start - 100)
        window_end = min(len(text), end + 100)
        window = text[window_start:window_end]

        best = None

        for m in self.email_regex.finditer(window):
            candidate_start = window_start + m.start()
            candidate_end = window_start + m.end()
            candidate_start, candidate_end = self.trim_email_span(
                text,
                candidate_start,
                candidate_end
            )

            if candidate_start <= start and candidate_end >= end:
                if best is None or (candidate_end - candidate_start) > (best[1] - best[0]):
                    best = (candidate_start, candidate_end)

        if best:
            return best

        return self.trim_email_span(text, start, end)

    def _add_entity(self, entities, text, start, end, source):
        start, end = self.extend_email_span(text, start, end)
        start, end = self.trim_email_span(text, start, end)

        if self.is_bad_email_candidate(text, start, end):
            return

        entities.append({
            "start": start,
            "end": end,
            "label": "EMAIL",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities,
                text,
                match.span.start,
                match.span.stop,
                "yargy_email",
            )

        for m in self.email_regex.finditer(text):
            self._add_entity(
                entities,
                text,
                m.start(),
                m.end(),
                "regex_email",
            )

        return resolve_overlaps(entities)



def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0

    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def run_yargy_email_experiment(path: str):
    df = load_dataset(path)
    detector = YargyEmailDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-EMAIL-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df


def print_email_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "EMAIL"]
        pred = [e for e in row["predictions"] if e["label"] == "EMAIL"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → EMAIL')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → EMAIL')

    return errors_df



path = "/content/test_v1 - test_final_300_v3.csv (20).csv"

summary, class_table, predictions_df = run_yargy_email_experiment(path)

predictions_df.to_csv("yargy_email_predictions.csv", index=False)
class_table.to_csv("yargy_email_metrics.csv", index=False)

errors_df = print_email_errors(predictions_df)
errors_df.to_csv("yargy_email_errors.csv", index=False)

In [ ]:
def evaluate_email_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)


    detector = YargyEmailDetector()
    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        email_preds = [p for p in preds if p["label"] == "EMAIL"]

        if email_preds:
            texts_with_fp += 1
            total_fp += len(email_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(email_preds),
                "false_positives": email_preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(f'FP: "{e["text"]}" [{e["start"]}:{e["end"]}] → {e["label"]}')

    return summary, errors_df

In [ ]:
nopd_email_summary, nopd_email_errors = evaluate_email_negative_or_tricky(
    "/content/no_pll_labeled - no_pll_labeled.csv (7).csv",
    name="no_pd"
)

nopd_email_errors.to_csv("yargy_email_no_pd_errors.csv", index=False)

In [ ]:
tricky_email_summary, tricky_email_errors = evaluate_email_negative_or_tricky(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv",
    name="tricky"
)

tricky_email_errors.to_csv("yargy_email_tricky_errors.csv", index=False)

ADDRESS

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df


def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])

class YargyAddressDetector:
    def __init__(self):
        EXTRA_PART = (
            r"(?:\s*,?\s*"
            r"(?:квартира|кв\.?|строение|стр\.?|корпус|корп\.?|офис|оф\.?|"
            r"владение|вл\.?|литер|лит\.?|к\.?)"
            r"\s*(?:\d{1,4}[А-Яа-яA-Za-z]?|[А-Яа-яA-Za-z])"
            r")*"
            r"(?:\s*,?\s*\d{1,2}\s*этаж)?"
        )

        STREET_TYPES = (
            r"ул\.?|улица|пр-кт|пр\.?|проспект|пр-т\.?|пер\.?|переулок|"
            r"наб\.?|б-р|бульвар|ш\.?|шоссе|тракт"
        )

        self.address_regexes = [
            r"(?:\d{6}\s*,\s*)?"
            r"(?:(?:Россия|МО|Московская\s+область|[А-ЯЁ][а-яё]+\s+(?:область|обл\.?|край|респ\.?|республика))\s*,\s*)?"
            r"(?:г\.?\s*)?[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,\s*"
            rf"(?:(?:{STREET_TYPES})\s*)?"
            r"[А-ЯЁA-Za-z0-9][А-ЯЁа-яёA-Za-z0-9\s\-]*?"
            r"\s*,\s*"
            r"(?:д\.?|дом)\s*\d{1,4}[А-Яа-яA-Za-z]?"
            + EXTRA_PART,

            r"(?:\d{6}\s*,\s*)?"
            r"(?:(?:Россия|МО|Московская\s+область|[А-ЯЁ][а-яё]+\s+(?:область|обл\.?|край|респ\.?|республика))\s*,\s*)?"
            r"г\.?\s*[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,\s*"
            r"[А-ЯЁ][А-ЯЁа-яёA-Za-z\-]*(?:\s+[А-ЯЁа-яёA-Za-z\-]+)?"
            r"\s*,\s*"
            r"(?:д\.?|дом)\s*\d{1,4}[А-Яа-яA-Za-z]?"
            + EXTRA_PART,

            r"г\.?\s*[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s+"
            rf"(?:(?:{STREET_TYPES})\s+)?"
            r"[А-ЯЁ][а-яёA-Za-z\-]+(?:\s+[А-ЯЁа-яёA-Za-z\-]+)?"
            r"\s+"
            r"(?:д\.?\s*)?\d{1,4}[А-Яа-яA-Za-z]?"
            r"(?:\s*(?:кв\.?|квартира|оф\.?|офис|стр\.?|строение|корп\.?|корпус|к\.?)"
            r"\s*(?:\d{1,4}[А-Яа-яA-Za-z]?|[А-Яа-яA-Za-z]))?",

            r"(?:адрес\s*:\s*)?"
            r"(?:г\.?\s*)?[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,?\s*"
            rf"(?:{STREET_TYPES})"
            r"\s*"
            r"[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,?\s*"
            r"(?:д\.?|дом)?\s*\d{1,4}[А-Яа-яA-Za-z]?"
            r"(?:\s*,?\s*(?:квартира|кв\.?|офис|оф\.?|строение|стр\.?|корпус|корп\.?|к\.?)"
            r"\s*(?:\d{1,4}[А-Яа-яA-Za-z]?|[А-Яа-яA-Za-z]))?",

            r"[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,\s*"
            rf"(?:{STREET_TYPES})"
            r"\s*[А-ЯЁ][а-яёA-Za-z\-]+"
            r"\s*,\s*д\.?\s*\d{1,4}[А-Яа-яA-Za-z]?"
            r"(?:\s*кв\.?\s*\d{1,4}[А-Яа-яA-Za-z]?)?",

            r"(?:(?:адрес|проживания|регистрации|корреспонденции|место\s+проживания)\s*:?\s*)"
            r"[А-ЯЁ][а-яёA-Za-z\-]+(?:\s+[А-ЯЁ][а-яёA-Za-z\-]+){0,3}"
            r"\s+\d{1,4}-\d{1,4}",

            r"[А-ЯЁ][а-яёA-Za-z\-]+(?:\s+[А-ЯЁа-яёA-Za-z\-]+){0,2}"
            r"\s+\d{1,4}[А-Яа-яA-Za-z]?"
            r"\s+(?:стр\.?|строение|корп\.?|корпус|к\.?|вл\.?|лит\.?|этаж)"
            r"\s*(?:\d{1,4}[А-Яа-яA-Za-z]?|[А-Яа-яA-Za-z])",

            r"[А-ЯЁ][а-яё\-]+\s+(?:обл\.?|область|край|респ\.?|республика)"
            r"\s*,\s*"
            r"[А-ЯЁ][а-яё\-]+\s+(?:р-н|район)"
            r"\s*,\s*"
            r"(?:пос\.?|пгт\.?|дер\.?|деревня|д\.?|с\.?|село)\s*[А-ЯЁ][а-яё\-]+"
            r"\s*,\s*"
            r"(?:д\.?|дом)\s*\d{1,4}[А-Яа-яA-Za-z]?",

            r"км\s*\d{1,4}\s+трассы\s+[A-ZА-ЯЁ]-?\d+\s+[А-ЯЁA-Zа-яёa-z\-]+"
            r"\s*,\s*"
            r"(?:владение|вл\.?)\s*\d{1,4}[А-Яа-яA-Za-z]?",

            r"\d{1,6}\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s+"
            r"(?:Street|St\.?|Avenue|Ave\.?|Road|Rd\.?)"
            r"\s*,\s*(?:Apt|Apartment)\s*\w+"
            r"\s*,\s*[A-Z][a-z]+"
            r"\s*,\s*[A-Z]{2}"
            r"\s*,\s*\d{5}"
            r"\s*,\s*USA",

            r"ul\.?\s+[A-Z][a-z]+"
            r"\s*,\s*d\.?\s*\d+"
            r"\s*,\s*kv\.?\s*\d+"
            r"\s*,\s*[A-Z][a-z]+"
            r"\s*,\s*Russia",

            r"(?:дом\s+)?напротив\s+школы\s*№?\s*\d+",
            r"рядом\s+с\s+ТЦ\s+[\"«'][^\"»']+[\"»']",

            r"в/ч\s*\d{3,10}\s*,?\s*общежитие\s*№?\s*\d+",
        ]

        self.parser = None

    def trim_address_span(self, text: str, start: int, end: int):
        while start < end and text[start].isspace():
            start += 1

        while end > start and text[end - 1] in ".,;:":
            end -= 1

        prefix_patterns = [
            r"^(?:фактический\s+адрес\s*:?\s*)",
            r"^(?:адрес\s+проживания\s*:?\s*)",
            r"^(?:адрес\s+регистрации\s*:?\s*)",
            r"^(?:адрес\s+для\s+корреспонденции\s*:?\s*)",
            r"^(?:адрес\s*:?\s*)",
            r"^(?:для\s+корреспонденции\s*:?\s*)",
            r"^(?:место\s+проживания\s*:?\s*)",
            r"^(?:места\s+проживания\s*:?\s*)",
            r"^(?:проживания\s*:?\s*)",
            r"^(?:регистрации\s*:?\s*)",
            r"^(?:корреспонденции\s*:?\s*)",
            r"^(?:этим\s+)",
            r"^(?:того\s+)",
        ]

        changed = True
        while changed:
            changed = False
            value = text[start:end]

            for pattern in prefix_patterns:
                m = re.match(pattern, value, flags=re.IGNORECASE)
                if m:
                    start += m.end()
                    changed = True
                    break

        return start, end

    def extend_address_left(self, text: str, start: int, end: int):
        left = text[max(0, start - 120):start]

        m = re.search(r"\d{6}\s*,\s*$", left)
        if m:
            start = start - (len(left) - m.start())

        left = text[max(0, start - 120):start]
        m = re.search(
            r"(?:Россия|МО|Московская\s+область|[А-ЯЁ][а-яё]+\s+(?:область|обл\.?|край|респ\.?|республика))\s*,\s*$",
            left,
            flags=re.IGNORECASE
        )
        if m:
            start = start - (len(left) - m.start())

        return start, end

    def extend_address_right(self, text: str, start: int, end: int):
        tail = text[end:end + 80]

        m = re.match(r"-\d{1,4}", tail)
        if m:
            end += m.end()
            tail = text[end:end + 80]

        m = re.match(
            r"(?:\s*,?\s*"
            r"(?:квартира|кв\.?|строение|стр\.?|корпус|корп\.?|офис|оф\.?|"
            r"владение|вл\.?|литер|лит\.?|к\.?)"
            r"\s*(?:\d{1,4}[А-Яа-яA-Za-z]?|[А-Яа-яA-Za-z])"
            r")+",
            tail,
            flags=re.IGNORECASE
        )

        if m:
            end += m.end()
            tail = text[end:end + 80]

        m = re.match(r"\s*,?\s*\d{1,2}\s*этаж", tail, flags=re.IGNORECASE)
        if m:
            end += m.end()
            tail = text[end:end + 80]

        value = text[start:end]
        bad_tail = re.search(
            r"(?:,\s*|\s+)(?:кр|кро|кром|кроме|корпу|корпус\s*$)$",
            value,
            flags=re.IGNORECASE
        )
        if bad_tail:
            end = start + bad_tail.start()

        value = text[start:end]
        stop = re.search(
            r"\s+(?:и|а также|наряду|вместе|дополнительно|номер|регистрационный|"
            r"идентификатор|телефон|email|почта|инн|договор|контракт|кроме|"
            r"используемые|предоставленные|подтвержденные|сохраненные|указанные)\b",
            value,
            flags=re.IGNORECASE
        )
        if stop:
            end = start + stop.start()

        while end > start and text[end - 1] in " ,.;:":
            end -= 1

        return start, end

    def is_bad_address_candidate(self, text: str, start: int, end: int) -> bool:
        value = text[start:end].strip()
        value_lower = value.lower()

        if len(value) < 5:
            return True

        if "гистрационный" in value_lower:
            return True

        if re.search(
            r"\b(?:контракт|контракта|контрактом|договор|договора|договором|заявка|"
            r"номер\s+документа|регистрационный\s+номер|id)\b",
            value_lower
        ):
            return True

        if re.search(
            r"\b(?:родился|родилась|рождения|года|лет|процент|процентов|секунды|"
            r"задержан|заместитель|министр|титул|турнире|календарь|яхту|стоимостью|"
            r"перевели|карта|карту|страны|главе|освободить|набрал)\b",
            value_lower
        ):
            return True

        left_context = text[max(0, start - 40):start].lower()
        if re.search(
            r"(?:контракт|контракта|контрактом|договор|договора|договором|"
            r"номер\s+документа|регистрационный\s+номер)\s*$",
            left_context
        ):
            return True

        if re.fullmatch(r"(?:д\.?|кв\.?|оф\.?|стр\.?|корп\.?|к\.?)\s*\d+", value_lower):
            return True

        markers = [
            "г.", "город", "ул", "улица", "проспект", "пр-кт", "пр-т", "пер", "переулок",
            "наб", "б-р", "тракт", "ш.", "шоссе", "дом", "д.", "кв", "оф",
            "стр", "строение", "корп", "к.", "обл", "край", "респ", "район", "р-н",
            "мо", "россия", "в/ч", "общежитие", "этаж", "владение", "литер",
            "street", "apt", "ul.", "kv.", "russia", "usa", "напротив", "рядом",
            "км", "трассы"
        ]

        has_marker = any(m in value_lower for m in markers)

        has_number_address = bool(
            re.search(r"[А-ЯЁA-Z][а-яёa-z\-]+\s+\d{1,4}(?:-\d{1,4})?", value)
        )

        if not has_marker and not has_number_address:
            return True

        return False

    def _add_entity(self, entities, text, start, end, source):
        start, end = self.trim_address_span(text, start, end)
        start, end = self.extend_address_left(text, start, end)
        start, end = self.extend_address_right(text, start, end)
        start, end = self.trim_address_span(text, start, end)

        if start >= end:
            return

        if self.is_bad_address_candidate(text, start, end):
            return

        entities.append({
            "start": start,
            "end": end,
            "label": "ADDRESS",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for pattern in self.address_regexes:
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                self._add_entity(
                    entities,
                    text,
                    m.start(),
                    m.end(),
                    "regex_address",
                )

        return resolve_overlaps(entities)


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0

    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)


def run_yargy_address_experiment(path: str):
    df = load_dataset(path)
    detector = YargyAddressDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-ADDRESS-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df



def print_address_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "ADDRESS"]
        pred = [e for e in row["predictions"] if e["label"] == "ADDRESS"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → ADDRESS')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → ADDRESS')

    return errors_df


path = "/content/test_v1 - test_final_300_v3.csv (20).csv"

summary, class_table, predictions_df = run_yargy_address_experiment(path)

predictions_df.to_csv("yargy_address_predictions.csv", index=False)
class_table.to_csv("yargy_address_metrics.csv", index=False)

errors_df = print_address_errors(predictions_df)
errors_df.to_csv("yargy_address_errors.csv", index=False)

In [ ]:
def evaluate_address_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)


    detector = YargyAddressDetector()
    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        address_preds = [p for p in preds if p["label"] == "ADDRESS"]

        if address_preds:
            texts_with_fp += 1
            total_fp += len(address_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(address_preds),
                "false_positives": address_preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(f'FP: "{e["text"]}" [{e["start"]}:{e["end"]}] → {e["label"]}')

    return summary, errors_df

In [ ]:
nopd_address_summary, nopd_address_errors = evaluate_address_negative_or_tricky(
    "/content/no_pll_labeled - no_pll_labeled.csv (7).csv",
    name="no_pd"
)

nopd_address_errors.to_csv("yargy_address_no_pd_errors.csv", index=False)

In [ ]:
tricky_address_summary, tricky_address_errors = evaluate_address_negative_or_tricky(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv",
    name="tricky"
)

tricky_address_errors.to_csv("yargy_address_tricky_errors.csv", index=False)

PERSON

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df


def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])


class YargyPersonDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def token_pred(pattern):
            return custom(lambda t: bool(re.fullmatch(pattern, token_value(t))))

        CAP_WORD = token_pred(r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?")
        INITIAL = token_pred(r"[А-ЯЁ]")

        PARTICLE = or_(
            rule(caseless("оглы")),
            rule(caseless("кызы")),
            rule(caseless("ибн")),
            rule(caseless("бен")),
            rule(caseless("бин")),
            rule(caseless("аль")),
            rule(caseless("де")),
            rule(caseless("ле")),
            rule(caseless("фон")),
            rule(caseless("ван")),
        )

        FULL_3 = rule(CAP_WORD, CAP_WORD, CAP_WORD)
        FULL_2 = rule(CAP_WORD, CAP_WORD)

        INITIALS_BEFORE = rule(
            INITIAL, eq("."),
            INITIAL.optional(), eq(".").optional(),
            CAP_WORD
        )

        INITIALS_AFTER = rule(
            CAP_WORD,
            INITIAL, eq("."),
            INITIAL.optional(), eq(".").optional()
        )

        PARTICLE_NAME = rule(
            CAP_WORD,
            CAP_WORD.optional(),
            PARTICLE,
            CAP_WORD.optional()
        )

        PERSON = or_(
            PARTICLE_NAME,
            INITIALS_BEFORE,
            INITIALS_AFTER,
            FULL_3,
            FULL_2,
        )

        self.parser = Parser(PERSON)

        self.person_regexes = [
            r"\b[А-ЯЁ]\s*\.\s*[А-ЯЁ]\s*\.?\s*"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ]\s*\.\s*"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ]\s*\.\s*[А-ЯЁ]\s*\.\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ]\.?\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"(?:оглы|кызы)\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"(?:ибн|бен|бин)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
            r"(?:\s+(?:Аль|аль)\s+[А-ЯЁ][а-яё]+)?"
            r"(?:\s+[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"(?:ибн|бен|бин)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"(?:аль|Аль)-[А-ЯЁ][а-яё]+\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"(?:де|ле|фон|ван|Ле|Ла|Де|Дю)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ][а-яё]+-[А-ЯЁ][а-яё]+\s+"
            r"(?:де\s+)?[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",

            r"\b(?:президент|министр|депутат|сенатор|мэр|судья|адвокат|прокурор|"
            r"губернатор|директор|руководитель|председатель|секретарь|"
            r"бизнесмен|предприниматель|космонавт|артист|актер|актриса|"
            r"тренер|спортсмен|журналист|писатель|поэт|генерал|"
            r"генерал-майор|генерал-лейтенант|полковник|вице-адмирал|"
            r"свидетель|обвиняемый|обвиняемая|подсудимый|подсудимая|"
            r"гражданин|гражданка|заявитель|заявительница|потерпевший|потерпевшая|"
            r"докладчик|докладчица|спикер|премьер|премьер-министр|"
            r"биатлонист|фигурист|математик|врач|певица|киноактриса|"
            r"житель|жителя|жительница|жительницу|эксперт|методист|"
            r"наркоторговец|заместитель)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
            r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?){0,2}\b",

            r"\b(?:отец|мать|сын|дочь|брат|сестра|супруг|супруга|жена|муж|"
            r"родитель|опекун|бабушка|дедушка|внучка|внук)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
            r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?){0,2}\b",

            r"\b(?:по словам|сообщил|сообщила|заявил|заявила|отметил|отметила|"
            r"рассказал|рассказала|прокомментировал|принял участие|дала интервью|"
            r"дал интервью|написал|написала|обсудил|спросил|ответил|"
            r"назначить|уволить|сам|бывший|новый)\s+"
            r"[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?"
            r"(?:\s+[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?){0,2}\b",
        ]

        self.prefix_strip_regex = re.compile(
            r"^(?:президент|министр|депутат|сенатор|мэр|судья|адвокат|прокурор|"
            r"губернатор|директор|руководитель|председатель|секретарь|"
            r"бизнесмен|предприниматель|космонавт|артист|актер|актриса|"
            r"тренер|спортсмен|журналист|писатель|поэт|генерал|"
            r"генерал-майор|генерал-лейтенант|полковник|вице-адмирал|"
            r"свидетель|обвиняемый|обвиняемая|подсудимый|подсудимая|"
            r"гражданин|гражданка|заявитель|заявительница|потерпевший|потерпевшая|"
            r"докладчик|докладчица|спикер|премьер|премьер-министр|"
            r"биатлонист|фигурист|математик|врач|певица|киноактриса|"
            r"житель|жителя|жительница|жительницу|эксперт|методист|"
            r"наркоторговец|заместитель|бывший|новый|сам|"
            r"назначить|уволить|"
            r"отец|мать|сын|дочь|брат|сестра|супруг|супруга|жена|муж|"
            r"родитель|опекун|бабушка|дедушка|внучка|внук|"
            r"по словам|сообщил|сообщила|заявил|заявила|отметил|отметила|"
            r"рассказал|рассказала|прокомментировал|принял участие|дала интервью|"
            r"дал интервью|написал|написала|обсудил|спросил|ответил)\s+",
            flags=re.IGNORECASE
        )

        self.bad_exact = {
            "Гран При",
            "Гран При Мексики",
            "Российская Федерация",
            "Российской Федерации",
            "Совет Федерации",
            "Совета Федерации",
            "Сбербанк Управление",
            "Сбербанк Управление Активами",
            "Пенсионный фонд",
            "РИА Новости",
            "Говорит Москва",
            "Golden Spin",
            "Fitness Balance",
            "Calories Tracker",
            "За права",
            "День судьи",
            "Пояснение Из",
            "На Кавминводах",
            "Шадринске Курганской",
            "Стратегия Будущего",
            "Московской Хельсинкской",
            "Войска Калининского",
            "Республики Бурятия",
            "Второй Опиумной",
            "Первым Чрезвычайным",
            "Верховным Главнокомандующим",
            "Совете Безопасности",
            "Южной Америки",
            "Южной Корее",
            "История Российская",
            "Дублина Графтон",
            "Эстонской Республике",
            "Академика Сахарова",
            "Генштаба Вооружённых",
        }

        self.bad_tokens = {
            "России", "Российской", "Федерации", "США", "РФ",
            "Минэнерго", "Минобороны", "Госдумы", "МВД", "КНДР",
            "МОК", "РАН", "АО", "ООО", "ЗАО", "ПАО",
            "Сбербанк", "Управление", "Активами",
            "Совета", "Федерации", "Эквадора", "Азербайджана",
            "Армении", "Катара", "Калининграда", "Вашингтон",
            "Москвы", "Парижа", "Сингапуре", "Тобольске",
            "Франции", "Венесуэлы", "Аргентины", "Минска",
            "Курганской", "Шадринске", "Кавминводах",
            "Республики", "Бурятия", "Саудовской", "Аравии",
            "Южной", "Америки", "Корее", "Баумана",
        }

    def trim_person_span(self, text: str, start: int, end: int):
        while start < end and text[start].isspace():
            start += 1

        while end > start and text[end - 1] in ",;:!?()[]«»\"'":
            end -= 1

        if end > start and text[end - 1] == ".":
            value = text[start:end]
            if not re.search(r"(?:[А-ЯЁ]\s*\.\s*){1,2}$", value):
                end -= 1

        changed = True
        while changed:
            changed = False
            value = text[start:end]
            m = self.prefix_strip_regex.match(value)
            if m:
                start += m.end()
                while start < end and text[start].isspace():
                    start += 1
                changed = True

        return start, end

    def strip_bad_left_tokens(self, text: str, start: int, end: int):
        value = text[start:end]
        tokens = list(re.finditer(r"\S+", value))

        while len(tokens) >= 2:
            first = tokens[0].group().strip(".,;:!?()[]«»\"'")
            if first in self.bad_tokens:
                start += tokens[0].end()
                while start < end and text[start].isspace():
                    start += 1

                value = text[start:end]
                tokens = list(re.finditer(r"\S+", value))
            else:
                break

        return start, end

    def is_bad_person_candidate(self, text: str, start: int, end: int) -> bool:
        value = text[start:end].strip()
        value_lower = value.lower()

        if len(value) < 2:
            return True

        if re.search(r"\d", value):
            return True

        if not re.search(r"[А-ЯЁ]", value):
            return True

        normalized_value = re.sub(r"\s+", " ", value)

        if normalized_value in self.bad_exact:
            return True

        bad_words = [
            "обращение", "заявление", "сведения", "анкета", "карточка",
            "адрес", "телефон", "почта", "email", "номер", "документ",
            "регистрационный", "контактные", "данные", "реквизиты",
            "улица", "проспект", "тракт", "дом", "квартира", "корпус",
            "сведения для", "дополнительные сведения", "фонд", "агентство",
            "министерство", "университет", "школа", "банк", "концерн",
            "правительство", "парламент", "суд", "область", "район",
            "войска", "республика", "республики", "история", "совет",
            "безопасности", "главнокомандующий", "главнокомандующим",
            "коллегия", "генштаб", "южной", "северной", "западной",
            "восточной",
        ]

        if any(x in value_lower for x in bad_words):
            return True

        if re.search(
            r"\b(?:ул|улица|проспект|пр-кт|пер|переулок|наб|б-р|дом|д\.|"
            r"кв|оф|корп|стр|обл|область|край|респ|район|р-н|инн|id|"
            r"ооо|ао|пао|зао|банк|фонд|университет|агентство|министерство|"
            r"совет|безопасности|войска|республика|генштаб|коллегия)\b",
            value_lower
        ):
            return True

        tokens = normalized_value.split()

        if len(tokens) > 1 and any(t in self.bad_tokens for t in tokens):
            return True

        if len(tokens) > 3 and not re.search(
            r"\b(?:оглы|кызы|ибн|бен|бин|аль|де|ле|фон|ван)\b",
            value_lower
        ):
            return True

        if len(tokens) == 1:
            left = text[max(0, start - 70):start].lower()
            right = text[end:min(len(text), end + 70)].lower()

            strong_context = re.search(
                r"(?:по словам|заявил|заявила|сообщил|сообщила|"
                r"отметил|отметила|рассказал|рассказала|"
                r"подозревается|обвинили|назначен|выступил|"
                r"выступила|уличил|пожаловался|вручил|"
                r"адвоката|жена|дочь|супруг|супруга|свидетель|"
                r"подсудимая|подсудимый|гражданин|гражданка|"
                r"россиянки|бразильца|принц|глава|лидера|имя)\s+$",
                left
            )

            strong_right = re.search(
                r"^\s*(?:уличил|пожаловался|занимала|выступил|"
                r"выступила|поддержал|создал|отрицает|"
                r"сообщил|сообщила|подтвердила|присутствовала|"
                r"вышел|на должность)",
                right
            )

            if not strong_context and not strong_right:
                return True

        return False

    def _add_entity(self, entities, text, start, end, source):
        start, end = self.trim_person_span(text, start, end)
        start, end = self.strip_bad_left_tokens(text, start, end)
        start, end = self.trim_person_span(text, start, end)

        if start >= end:
            return

        if self.is_bad_person_candidate(text, start, end):
            return

        entities.append({
            "start": start,
            "end": end,
            "label": "PERSON",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities,
                text,
                match.span.start,
                match.span.stop,
                "yargy_person",
            )

        for pattern in self.person_regexes:
            for m in re.finditer(pattern, text):
                self._add_entity(
                    entities,
                    text,
                    m.start(),
                    m.end(),
                    "regex_person",
                )

        return resolve_overlaps(entities)


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)


def run_yargy_person_experiment(path: str):
    df = load_dataset(path)
    detector = YargyPersonDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-PERSON-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df


def print_person_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "PERSON"]
        pred = [e for e in row["predictions"] if e["label"] == "PERSON"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → PERSON')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → PERSON')

    return errors_df


path = "/content/test_v1 - test_final_300_v3.csv (20).csv"

summary, class_table, predictions_df = run_yargy_person_experiment(path)

predictions_df.to_csv("yargy_person_predictions.csv", index=False)
class_table.to_csv("yargy_person_metrics.csv", index=False)

errors_df = print_person_errors(predictions_df)
errors_df.to_csv("yargy_person_errors.csv", index=False)

In [ ]:
def evaluate_person_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)

    detector = YargyPersonDetector()
    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        person_preds = [p for p in preds if p["label"] == "PERSON"]

        if person_preds:
            texts_with_fp += 1
            total_fp += len(person_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(person_preds),
                "false_positives": person_preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(f'FP: "{e["text"]}" [{e["start"]}:{e["end"]}] → {e["label"]}')

    return summary, errors_df

In [ ]:
nopd_person_summary, nopd_person_errors = evaluate_person_negative_or_tricky(
    "/content/no_pll_labeled - no_pll_labeled.csv (7).csv",
    name="no_pd"
)

nopd_person_errors.to_csv("yargy_person_no_pd_errors.csv", index=False)

In [ ]:
tricky_person_summary, tricky_person_errors = evaluate_person_negative_or_tricky(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv",
    name="tricky"
)

tricky_person_errors.to_csv("yargy_person_tricky_errors.csv", index=False)

ОБЪЕДИНЕНИЕ КЛАССОВ

In [ ]:
import ast
import time
import pandas as pd


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")

        if "text" not in df.columns:
            df = pd.read_csv(path)

    except Exception:
        df = pd.read_csv(path)

    if "label" not in df.columns:
        df["label"] = [[] for _ in range(len(df))]
    else:
        df["label"] = df["label"].fillna("[]")

    return df


def normalize_gold(label_list):

    if label_list is None:
        return []

    if isinstance(label_list, float):
        return []

    if isinstance(label_list, str):
        label_list = label_list.strip()

        if label_list == "" or label_list == "[]":
            return []

        try:
            label_list = ast.literal_eval(label_list)
        except Exception:
            return []

    if not isinstance(label_list, list):
        return []

    result = []

    for x in label_list:

        if not isinstance(x, (list, tuple)):
            continue

        if len(x) < 3:
            continue

        result.append({
            "start": int(x[0]),
            "end": int(x[1]),
            "label": str(x[2]),
        })

    return result


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps_global(entities):

    priority = {
        "EMAIL": 5,
        "PHONE": 4,
        "ID": 3,
        "ADDRESS": 2,
        "PERSON": 1,
    }

    entities = sorted(
        entities,
        key=lambda e: (
            e["start"],
            -(e["end"] - e["start"]),
            -priority.get(e["label"], 0)
        )
    )

    selected = []

    for ent in entities:
        keep = True

        for old in selected[:]:
            overlap = not (
                ent["end"] <= old["start"] or ent["start"] >= old["end"]
            )

            if overlap:
                ent_len = ent["end"] - ent["start"]
                old_len = old["end"] - old["start"]

                ent_score = (ent_len, priority.get(ent["label"], 0))
                old_score = (old_len, priority.get(old["label"], 0))

                if ent_score > old_score:
                    selected.remove(old)
                else:
                    keep = False

                break

        if keep:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])


class RuleBasedExtractor:
    def __init__(self):
        self.detectors = {
            "PHONE": YargyPhoneDetector(),
            "EMAIL": YargyEmailDetector(),
            "ID": YargyIdDetector(),
            "ADDRESS": YargyAddressDetector(),
            "PERSON": YargyPersonDetector(),
        }

    def predict_one(self, text: str):
        entities = []

        for label, detector in self.detectors.items():
            preds = detector.predict_one(text)

            for ent in preds:
                ent = dict(ent)
                ent["label"] = label

                if "text" not in ent:
                    ent["text"] = text[ent["start"]:ent["end"]]

                ent["source_detector"] = label
                entities.append(ent)

        return resolve_overlaps_global(entities)



def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def build_errors_by_class(predictions_df: pd.DataFrame, cls: str):
    rows = []

    for _, row in predictions_df.iterrows():
        text = row["text"]

        gold = [e for e in row["gold"] if e["label"] == cls]
        pred = [e for e in row["predictions"] if e["label"] == cls]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": text,
                "class": cls,
                "false_positive": fp,
                "false_negative": fn,
            })

    return pd.DataFrame(rows)


def print_errors_by_class(errors_df: pd.DataFrame, cls: str, limit=None):
    print(f"Количество текстов с ошибками: {len(errors_df)}")

    shown = errors_df if limit is None else errors_df.head(limit)

    for _, row in shown.iterrows():
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e.get("text", row["text"][e["start"]:e["end"]])}" [{e["start"]}:{e["end"]}] → {e["label"]}')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → {e["label"]}')



def run_rule_based_experiment(path: str, dataset_name: str = "dataset", print_error_limit=None):
    df = load_dataset(path)
    extractor = RuleBasedExtractor()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [extractor.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "dataset": dataset_name,
        "method": "Rule-based unified",
        "texts_total": len(df),
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(f"DATASET: {dataset_name}")
    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    errors_by_class = {}

    for cls in ENTITY_CLASSES:
        errors_df = build_errors_by_class(predictions_df, cls)
        errors_by_class[cls] = errors_df
        print_errors_by_class(errors_df, cls, limit=print_error_limit)

    return summary, class_table, predictions_df, errors_by_class

In [ ]:
train_path = "/content/train_with_hard_no_pd.csv"
test_path = "/content/test_small - test_final_300_v3.csv.csv"


train_summary, train_class_table, train_predictions_df, train_errors_by_class = run_rule_based_experiment(
    train_path,
    dataset_name="train",
    print_error_limit=20
)

In [ ]:
test_summary, test_class_table, test_predictions_df, test_errors_by_class = run_rule_based_experiment(
    test_path,
    dataset_name="test",
    print_error_limit=20
)

In [ ]:
train_class_table.to_csv("rule_based_train_metrics_by_class.csv", index=False)
test_class_table.to_csv("rule_based_test_metrics_by_class.csv", index=False)

train_predictions_df.to_csv("rule_based_train_predictions.csv", index=False)
test_predictions_df.to_csv("rule_based_test_predictions.csv", index=False)

for cls, errors_df in train_errors_by_class.items():
    errors_df.to_csv(f"rule_based_train_errors_{cls}.csv", index=False)

for cls, errors_df in test_errors_by_class.items():
    errors_df.to_csv(f"rule_based_test_errors_{cls}.csv", index=False)

In [ ]:
def evaluate_rule_based_negative_or_tricky(path: str, name: str = "negative_set"):
    df = pd.read_csv(path)


    extractor = RuleBasedExtractor()
    texts = df["text"].astype(str).tolist()

    start_time = time.time()
    predictions = [extractor.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    rows = []
    total_fp = 0
    texts_with_fp = 0

    fp_by_class = {cls: 0 for cls in ENTITY_CLASSES}
    texts_with_fp_by_class = {cls: 0 for cls in ENTITY_CLASSES}

    for i, (text, preds) in enumerate(zip(texts, predictions)):
        if preds:
            texts_with_fp += 1
            total_fp += len(preds)

            for cls in ENTITY_CLASSES:
                cls_preds = [p for p in preds if p["label"] == cls]
                if cls_preds:
                    texts_with_fp_by_class[cls] += 1
                    fp_by_class[cls] += len(cls_preds)

            rows.append({
                "id": i,
                "text": text,
                "entity_hint": df.loc[i, "entity"] if "entity" in df.columns else None,
                "bucket": df.loc[i, "bucket"] if "bucket" in df.columns else None,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "false_positive_count": len(preds),
                "false_positives": preds,
            })

    summary = {
        "dataset": name,
        "texts_total": len(df),
        "texts_with_fp": texts_with_fp,
        "total_fp": total_fp,
        "text_error_rate": round(texts_with_fp / len(df), 4) if len(df) else 0,
        "fp_per_100_texts": round(total_fp / len(df) * 100, 2) if len(df) else 0,
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    class_rows = []
    for cls in ENTITY_CLASSES:
        class_rows.append({
            "Класс": cls,
            "texts_with_fp": texts_with_fp_by_class[cls],
            "total_fp": fp_by_class[cls],
            "text_error_rate": round(texts_with_fp_by_class[cls] / len(df), 4) if len(df) else 0,
            "fp_per_100_texts": round(fp_by_class[cls] / len(df) * 100, 2) if len(df) else 0,
        })

    class_table = pd.DataFrame(class_rows)
    errors_df = pd.DataFrame(rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    print(f"Текстов с ошибками: {texts_with_fp}")
    print(f"Всего FP: {total_fp}")

    for _, row in errors_df.iterrows():
        print(f"ID: {row['id']}")

        if row.get("entity_hint"):
            print(f"Entity hint: {row['entity_hint']}")

        if row.get("bucket"):
            print(f"Bucket: {row['bucket']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        for e in row["false_positives"]:
            print(
                f'FP: "{e.get("text", row["text"][e["start"]:e["end"]])}" '
                f'[{e["start"]}:{e["end"]}] → {e["label"]}'
            )

    return summary, class_table, errors_df

In [ ]:
nopd_summary, nopd_class_table, nopd_errors = evaluate_rule_based_negative_or_tricky(
    "/content/no_pd_eval_clean.csv",
    name="no_pd"
)



In [ ]:
import ast
import time
import pandas as pd
from tqdm.auto import tqdm


def parse_labels(x):
    if isinstance(x, list):
        return x
    if pd.isna(x) or str(x).strip() in ["", "[]"]:
        return []
    return ast.literal_eval(str(x))


def normalize_entities(label_list, text=None):
    result = []

    for x in label_list:
        start = int(x[0])
        end = int(x[1])
        label = x[2]

        ent = {
            "start": start,
            "end": end,
            "label": label,
        }

        if text is not None:
            ent["text"] = text[start:end]

        result.append(ent)

    return result


def entity_key(ent):
    return (int(ent["start"]), int(ent["end"]), ent["label"])


def spans_overlap(a, b):
    return not (a["end"] <= b["start"] or a["start"] >= b["end"])


def overlap_len(a, b):
    return max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))


def classify_errors_for_text(text, gold, pred):
    errors = []

    matched_gold = set()
    matched_pred = set()

    for gi, g in enumerate(gold):
        for pi, p in enumerate(pred):
            if pi in matched_pred:
                continue

            if entity_key(g) == entity_key(p):
                matched_gold.add(gi)
                matched_pred.add(pi)
                break

    for gi, g in enumerate(gold):
        if gi in matched_gold:
            continue

        overlapping = [
            (pi, p)
            for pi, p in enumerate(pred)
            if pi not in matched_pred and spans_overlap(g, p)
        ]

        if not overlapping:
            errors.append({
                "error_type": "FN",
                "gold": g,
                "pred": None,
                "gold_text": text[g["start"]:g["end"]],
                "pred_text": None,
            })
            matched_gold.add(gi)
            continue

        pi, p = max(overlapping, key=lambda x: overlap_len(g, x[1]))

        same_span = g["start"] == p["start"] and g["end"] == p["end"]
        same_label = g["label"] == p["label"]

        if same_span and not same_label:
            error_type = "WRONG_LABEL"
        elif not same_span and same_label:
            error_type = "BOUNDARY_ERROR"
        else:
            error_type = "WRONG_LABEL_AND_BOUNDARY"

        errors.append({
            "error_type": error_type,
            "gold": g,
            "pred": p,
            "gold_text": text[g["start"]:g["end"]],
            "pred_text": p.get("text", text[p["start"]:p["end"]]),
        })

        matched_gold.add(gi)
        matched_pred.add(pi)

    for pi, p in enumerate(pred):
        if pi not in matched_pred:
            errors.append({
                "error_type": "FP",
                "gold": None,
                "pred": p,
                "gold_text": None,
                "pred_text": p.get("text", text[p["start"]:p["end"]]),
            })

    return errors


def evaluate_rule_based_tricky_detailed(path: str, name: str = "tricky", max_print=None):
    df = pd.read_csv(path)


    if "label" in df.columns:
        df["label"] = df["label"].apply(parse_labels)
    else:
        df["label"] = [[] for _ in range(len(df))]

    extractor = RuleBasedExtractor()

    all_error_rows = []
    prediction_rows = []

    error_counts = {
        "FP": 0,
        "FN": 0,
        "WRONG_LABEL": 0,
        "BOUNDARY_ERROR": 0,
        "WRONG_LABEL_AND_BOUNDARY": 0,
    }

    texts_with_errors = 0

    start_time = time.time()

    for i, row in tqdm(df.reset_index(drop=True).iterrows(), total=len(df), desc=name):
        text = str(row["text"])
        gold = normalize_entities(row["label"], text=text)
        pred = extractor.predict_one(text)

        for p in pred:
            p["start"] = int(p["start"])
            p["end"] = int(p["end"])
            p["text"] = p.get("text", text[p["start"]:p["end"]])

        errors = classify_errors_for_text(text, gold, pred)

        prediction_rows.append({
            "id": i,
            "text": text,
            "gold": gold,
            "predictions": pred,
            "errors": errors,
            "has_error": bool(errors),
        })

        if errors:
            texts_with_errors += 1

            for e in errors:
                error_counts[e["error_type"]] += 1

                all_error_rows.append({
                    "id": i,
                    "source": row.get("source", ""),
                    "bucket": row.get("bucket", ""),
                    "entity_hint": row.get("entity", ""),
                    "text": text,
                    "error_type": e["error_type"],

                    "gold_text": e["gold_text"],
                    "gold_start": None if e["gold"] is None else e["gold"]["start"],
                    "gold_end": None if e["gold"] is None else e["gold"]["end"],
                    "gold_label": None if e["gold"] is None else e["gold"]["label"],

                    "pred_text": e["pred_text"],
                    "pred_start": None if e["pred"] is None else e["pred"]["start"],
                    "pred_end": None if e["pred"] is None else e["pred"]["end"],
                    "pred_label": None if e["pred"] is None else e["pred"]["label"],
                    "pred_source": None if e["pred"] is None else e["pred"].get("source", e["pred"].get("source_component", "")),
                })

    elapsed = time.time() - start_time

    errors_df = pd.DataFrame(all_error_rows)
    predictions_df = pd.DataFrame(prediction_rows)

    summary = {
        "dataset": name,
        "method": "rule-based",
        "texts_total": len(df),
        "texts_with_errors": texts_with_errors,
        "text_error_rate": round(texts_with_errors / len(df), 4) if len(df) else 0,
        "total_errors": len(errors_df),
        "FP": error_counts["FP"],
        "FN": error_counts["FN"],
        "WRONG_LABEL": error_counts["WRONG_LABEL"],
        "BOUNDARY_ERROR": error_counts["BOUNDARY_ERROR"],
        "WRONG_LABEL_AND_BOUNDARY": error_counts["WRONG_LABEL_AND_BOUNDARY"],
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    summary_df = pd.DataFrame([summary])

    if len(errors_df):
        by_class = (
            errors_df
            .assign(class_for_count=lambda x: x["pred_label"].fillna(x["gold_label"]))
            .groupby(["class_for_count", "error_type"])
            .size()
            .reset_index(name="count")
            .sort_values(["class_for_count", "error_type"])
        )
    else:
        by_class = pd.DataFrame(columns=["class_for_count", "error_type", "count"])

    print(summary_df.to_string(index=False))

    if len(errors_df):
        print(errors_df["error_type"].value_counts().to_string())
    else:
        print("Ошибок нет")

    print(by_class.to_string(index=False))

    rows_to_print = errors_df if max_print is None else errors_df.head(max_print)

    for _, r in rows_to_print.iterrows():
        print(f"ID: {r['id']}")
        if r.get("source", ""):
            print(f"Source: {r['source']}")
        if r.get("bucket", ""):
            print(f"Bucket: {r['bucket']}")
        if r.get("entity_hint", ""):
            print(f"Entity hint: {r['entity_hint']}")

        print("Текст:", r["text"])
        print("Ошибка:", r["error_type"])

        if r["error_type"] == "FP":
            print(f'FP: "{r["pred_text"]}" [{r["pred_start"]}:{r["pred_end"]}] → {r["pred_label"]}')

        elif r["error_type"] == "FN":
            print(f'FN: "{r["gold_text"]}" [{r["gold_start"]}:{r["gold_end"]}] → {r["gold_label"]}')

        else:
            print(f'GOLD: "{r["gold_text"]}" [{r["gold_start"]}:{r["gold_end"]}] → {r["gold_label"]}')
            print(f'PRED: "{r["pred_text"]}" [{r["pred_start"]}:{r["pred_end"]}] → {r["pred_label"]}')

    return summary_df, by_class, errors_df, predictions_df

In [ ]:
tricky_summary_df, tricky_by_class_df, tricky_errors_df, tricky_predictions_df = (
    evaluate_rule_based_tricky_detailed(
        "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (6).csv",
        name="tricky",
        max_print=None
    )
)

tricky_summary_df.to_csv("rule_based_tricky_summary_detailed.csv", index=False)
tricky_by_class_df.to_csv("rule_based_tricky_errors_by_class.csv", index=False)
tricky_errors_df.to_csv("rule_based_tricky_errors_detailed.csv", index=False)
tricky_predictions_df.to_csv("rule_based_tricky_predictions_detailed.csv", index=False)